In [ ]:
import os
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import albumentations as A
from albumentations.pytorch import ToTensorV2
from PIL import Image
import numpy as np
from sklearn.preprocessing import LabelEncoder
from tqdm import tqdm
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report
import io
import hashlib
from pathlib import Path
import gc
from sklearn.model_selection import train_test_split
import random
from collections import Counter
import tempfile


c:\ITBA\MINICONDA\envs\TrabajoPractico_1\Lib\site-packages\albumentations\__init__.py:28: UserWarning: A new version of Albumentations is available: '2.0.8' (you have '2.0.7'). Upgrade using: pip install -U albumentations. To disable automatic update checks, set the environment variable NO_ALBUMENTATIONS_UPDATE to 1.
  check_for_updates()


In [ ]:
import mlflow
import mlflow.pytorch

In [ ]:
# Creamos el "experimento" en MLflow
mlflow.set_experiment("MLP_Clasificador_Imagenes")

<Experiment: artifact_location='file:///c:/ITBA/REDES%20NEURONALES/Tp1-Redes-Neuronales/mlruns/548689065550430374', creation_time=1779236188962, experiment_id='548689065550430374', last_update_time=1779236188962, lifecycle_stage='active', name='MLP_Clasificador_Imagenes', tags={}>

In [ ]:
from torch.utils.tensorboard import SummaryWriter
import torchvision.utils as vutils

In [ ]:
# Función para loguear una figura matplotlib en TensorBoard
def plot_to_tensorboard(fig, writer, tag, step):
    buf = io.BytesIO()
    fig.savefig(buf, format='png')
    buf.seek(0)
    image = Image.open(buf).convert("RGB")
    image = np.array(image)
    image = torch.tensor(image).permute(2, 0, 1) / 255.0
    writer.add_image(tag, image, global_step=step)
    plt.close(fig)

In [ ]:
def log_classification_report(model, loader, writer, device, classes, step, prefix="val"):
    model.eval()
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            outputs = model(images)
            _, preds = torch.max(outputs, 1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.numpy())

    all_labels = np.array(all_labels)
    all_preds = np.array(all_preds)

    # Definimos los índices fijos de todas las clases posibles
    all_class_indices = list(range(len(classes)))

    # Calculamos la matriz con tamaño fijo (siempre mapeando todas las clases)
    cm = confusion_matrix(all_labels, all_preds, labels=all_class_indices)
    
    fig_cm, ax = plt.subplots(figsize=(8, 8)) # Un poco más grande por ser varias clases
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=classes)
    disp.plot(ax=ax, cmap='Blues', xticks_rotation=45)
    ax.set_title(f'{prefix.title()} - Confusion Matrix (Epoch {step})')
    plt.tight_layout()

    fig_path = f"confusion_matrix_{prefix}_epoch_{step}.png"
    fig_cm.savefig(fig_path)
    mlflow.log_artifact(fig_path)
    
    # Mandamos a TensorBoard
    plot_to_tensorboard(fig_cm, writer, f"{prefix}/confusion_matrix", step)
    
    try:
        if os.path.exists(fig_path):
            os.remove(fig_path)
    except Exception:
        pass

    # Generamos el reporte 
    cls_report = classification_report(all_labels, all_preds, target_names=classes, labels=all_class_indices, zero_division=0)
    writer.add_text(f"{prefix}/classification_report", f"<pre>{cls_report}</pre>", step)

    report_path = f"classification_report_{prefix}_epoch_{step}.txt"
    with open(report_path, "w") as f:
        f.write(cls_report)
    mlflow.log_artifact(report_path)
    
    try:
        if os.path.exists(report_path):
            os.remove(report_path)
    except Exception:
        pass

In [ ]:
# Crear directorio de logs
log_dir = "runs/mlp_experimento_1"
writer = SummaryWriter(log_dir=log_dir)

In [ ]:
# Clase que le dice a PyTorch cómo leer nuestras imágenes, recorre las carpetas, asocia cada imagen con su clase, y aplica los transforms (resize, augmentations, normalización)

class CustomImageDataset(Dataset):
    def __init__(self, image_paths, transform=None):
        self.image_paths = image_paths
        self.transform = transform
        
        self.classes = sorted(list(set([Path(p).parent.name for p in self.image_paths])))
        self.class_to_idx = {cls_name: idx for idx, cls_name in enumerate(self.classes)}
        self.labels = [self.class_to_idx[Path(p).parent.name] for p in self.image_paths]

    def __len__(self):
        return len(self.image_paths)
    
    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        image = np.array(Image.open(img_path).convert("RGB"))
        label = self.labels[idx]

        if self.transform:
            augmented = self.transform(image=image)
            image = augmented["image"]

        return image, label

In [ ]:

train_transform = A.Compose([
    A.Resize(32, 32),

    # GEOMÉTRICAS MÉDICAS (Rotación total indispensable)
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.RandomRotate90(p=0.5),

    # CONTRASTE MÉDICO (Muy útil en dermatoscopía para resaltar la lesión)
    A.CLAHE(clip_limit=2.0, p=0.3), 
    A.RandomBrightnessContrast(brightness_limit=0.1, contrast_limit=0.1, p=0.3),

    # COLOR LEVE (Simula variaciones de luz/cámara entre consultorios)
    A.HueSaturationValue(hue_shift_limit=5, sat_shift_limit=10, val_shift_limit=5, p=0.2),

    A.Normalize(),
    ToTensorV2()
])

In [ ]:
# TRANSFORMS DE VAL: sin augmentations, solo resize y normalizar (no queremos modificar las imágenes de validación)

val_test_transform = A.Compose([
    A.Resize(32, 32),
    A.Normalize(),
    ToTensorV2()
])

In [ ]:
# JUNTAMOS LAS FOTOS
data_dir_total = r'data/Split_smol/'
valid_extensions = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}

def get_class(x): 
    return x.parent.name

files_totales = []

# Buscamos recursivamente en todas las subcarpetas
for x in Path(data_dir_total).rglob('*'):
    if x.is_file() and x.suffix.lower() in valid_extensions:
        try:
            with Image.open(x) as img:
                files_totales.append((x, get_class(x), img.size, img.mode))
        except Exception:
            pass 

# Creamos el DataFrame original
df_completo = pd.DataFrame(files_totales, columns=["path", "class", "resolution", "mode"])
print(f"Total de imágenes encontradas en bruto: {len(df_completo)}")

# Función auxiliar para calcular el hash MD5
def calcular_md5(path_objeto):
    hash_md5 = hashlib.md5()
    with open(path_objeto, "rb") as f:
        for chunk in iter(lambda: f.read(4096), b""):
            hash_md5.update(chunk)
    return hash_md5.hexdigest()

# Calculamos el hash para cada foto
df_completo['md5'] = df_completo['path'].apply(calcular_md5)

# Sacamos fotos problemáticas por nombre
fotos_a_eliminar = {"aug_0_F2.large.jpg"}  #(foto negra)
df_completo = df_completo[~df_completo['path'].apply(lambda p: p.name).isin(fotos_a_eliminar)].reset_index(drop=True)
print(f"Total después de eliminar fotos problemáticas: {len(df_completo)}")

# Borramos los duplicados basándonos en el hash
df_limpio = df_completo.drop_duplicates(subset=['md5'], keep='first').reset_index(drop=True)
print(f"Total de imágenes después de eliminar duplicados: {len(df_limpio)}")

# PASO 1: Separamos el 20% para el TEST FINAL
df_train_val, df_test = train_test_split(
    df_limpio, 
    test_size=0.20, 
    stratify=df_limpio['class'], 
    random_state=42
)

# PASO 2: Del 80% restante, separamos el 25% para VALIDACIÓN
df_train, df_val = train_test_split(
    df_train_val, 
    test_size=0.25, 
    stratify=df_train_val['class'], 
    random_state=42
)

df_train = df_train.reset_index(drop=True)
df_val   = df_val.reset_index(drop=True)
df_test  = df_test.reset_index(drop=True)

train_image_paths = df_train["path"].apply(lambda p: str(p)).tolist()
val_image_paths   = df_val["path"].apply(lambda p: str(p)).tolist()
test_image_paths  = df_test["path"].apply(lambda p: str(p)).tolist()

print("\n--- Split 60/20/20 ---")
print(f"Train (60%): {len(train_image_paths)}")
print(f"Val   (20%): {len(val_image_paths)}")
print(f"Test  (20%): {len(test_image_paths)}")

Total de imágenes encontradas en bruto: 876
Total después de eliminar fotos problemáticas: 875
Total de imágenes después de eliminar duplicados: 842

--- Split 60/20/20 ---
Train (60%): 504
Val   (20%): 169
Test  (20%): 169


In [ ]:
# Calculamos cuántas imágenes tiene cada clase en train
counts = Counter([Path(p).parent.name for p in train_image_paths])
max_count = max(counts.values())

# Rellenamos cada clase hasta llegar al máximo
for cls, count in counts.items():
    faltantes = max_count - count
    if faltantes > 0:
        paths_cls = [p for p in train_image_paths if Path(p).parent.name == cls]
        train_image_paths.extend(random.choices(paths_cls, k=faltantes))

print(f"Total de imágenes en TRAIN después del oversampling: {len(train_image_paths)}")

# Verificamos que quedó balanceado
counts_post = Counter([Path(p).parent.name for p in train_image_paths])
for cls, count in sorted(counts_post.items()):
    print(f"  {cls}: {count}")


# Forzar limpieza en Jupyter
if 'train_dataset' in locals(): del train_dataset
if 'val_dataset' in locals(): del val_dataset
if 'test_dataset' in locals(): del test_dataset
gc.collect()

train_dataset = CustomImageDataset(train_image_paths, transform=train_transform)
val_dataset   = CustomImageDataset(val_image_paths,   transform=val_test_transform)
test_dataset  = CustomImageDataset(test_image_paths,  transform=val_test_transform)

batch_size = 32
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=batch_size)
test_loader  = DataLoader(test_dataset,  batch_size=batch_size)

print(f"DataLoaders listos de forma limpia:")
print(f"Train: {len(train_dataset)} | Val: {len(val_dataset)} | Test: {len(test_dataset)}")

Total de imágenes en TRAIN después del oversampling: 540
  Actinic keratosis: 60
  Atopic Dermatitis: 60
  Benign keratosis: 60
  Dermatofibroma: 60
  Melanocytic nevus: 60
  Melanoma: 60
  Squamous cell carcinoma: 60
  Tinea Ringworm Candidiasis: 60
  Vascular lesion: 60
DataLoaders listos de forma limpia:
Train: 540 | Val: 169 | Test: 169


In [ ]:
class MLPClassifier(nn.Module):
    
    def __init__(
        self,
        num_classes,
        input_size=32*32*3,
        dropout_rate_1=0.25,
        dropout_rate_2=0.0
    ):

        super().__init__()

        self.model = nn.Sequential(

            nn.Flatten(),

            # Capa 1
            nn.Linear(input_size, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(dropout_rate_1),

            # Capa 2
            nn.Linear(512, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(dropout_rate_2),

            # Output
            nn.Linear(128, num_classes)
        )

        self.init_weights()

    def init_weights(self):

        for m in self.modules():

            if isinstance(m, nn.Linear):

                nn.init.kaiming_normal_(m.weight)
                nn.init.zeros_(m.bias)

    def forward(self, x):

        return self.model(x)
            

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
num_classes = len(train_dataset.classes)
model = MLPClassifier(num_classes=num_classes).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=3e-4, weight_decay=1e-4)


In [ ]:
def evaluate(model, loader, epoch=None, prefix="val"):
    model.eval()  #  primero esto, siempre
    model.to(device) 
    
    
    log_classification_report(model, loader, writer, device, train_dataset.classes, step=epoch, prefix=prefix)
    
    correct, total, loss_sum = 0, 0, 0.0
    all_preds, all_labels = [], []

    
    if 'criterion' in globals():
        global criterion
        criterion = criterion.to(device)

    with torch.no_grad():
        for i, (images, labels) in enumerate(loader):
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            _, preds = torch.max(outputs, 1)

            loss_sum += loss.item()
            correct  += (preds == labels).sum().item()
            total    += labels.size(0)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

            if i == 0 and epoch is not None:
                img_grid = vutils.make_grid(images[:8].cpu(), normalize=True)
                writer.add_image(f"{prefix}/images", img_grid, global_step=epoch)

    acc      = 100.0 * correct / total
    avg_loss = loss_sum / len(loader)

    if epoch is not None:
        writer.add_scalar(f"{prefix}/loss",     avg_loss, epoch)
        writer.add_scalar(f"{prefix}/accuracy", acc,      epoch)

    return avg_loss, acc

In [ ]:
n_epochs = 60  # ponemos alto, el ES lo para antes
es_patience = 5
best_val_loss = float('inf')
epochs_sin_mejora = 0

# Definís los dos valores independientes antes de correr
dropout_1 = 0.25
dropout_2 = 0.0

# Registro en MLflow
with mlflow.start_run():
    mlflow.log_params({
        "model": "MLPClassifier",
        "input_size": 32*32*3,
        "batch_size": batch_size,
        "lr": 3e-4,
        "epochs": n_epochs,
        "es_patience": es_patience,
        "optimizer": "Adam",
        "weight_decay": 1e-4,
        "batch_norm": True,
        "dropout_1": dropout_1,  # <--- Guardás el de la capa 1
        "dropout_2": dropout_2,  # <--- Guardás el de la capa 2
        "loss_fn": "CrossEntropyLoss",
        "data_dir": data_dir_total,
        "n_train": len(train_image_paths),
        "n_val": len(val_image_paths),
    })

    for epoch in range(n_epochs):
        model.train()
        running_loss = 0.0
        correct, total = 0, 0

        for images, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}/{n_epochs}"):
            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()
            _, preds = torch.max(outputs, 1)
            correct += (preds == labels).sum().item()
            total   += labels.size(0)

        train_loss = running_loss / len(train_loader)
        train_acc  = 100.0 * correct / total
        val_loss, val_acc = evaluate(model, val_loader, epoch=epoch, prefix="val")

        print(f"Epoch {epoch+1}:")
        print(f"  Train Loss: {train_loss:.4f}, Accuracy: {train_acc:.2f}%")
        print(f"  Val   Loss: {val_loss:.4f}, Accuracy: {val_acc:.2f}%")

        writer.add_scalar("train/loss",     train_loss, epoch)
        writer.add_scalar("train/accuracy", train_acc,  epoch)

        mlflow.log_metrics({
            "train_loss":     train_loss,
            "train_accuracy": train_acc,
            "val_loss":       val_loss,
            "val_accuracy":   val_acc
        }, step=epoch)

        # Early stopping
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            epochs_sin_mejora = 0
            
            
            tmp_path = f"best_model_tmp_{os.getpid()}.pth"
            torch.save(model.state_dict(), tmp_path)
            mlflow.log_artifact(tmp_path)
            os.remove(tmp_path)
            
            
        else:
            epochs_sin_mejora += 1
            if epochs_sin_mejora >= es_patience:
                print(f"Early stopping en época {epoch+1}")
                break

    print("Modelo guardado como 'mlp_model.pth'")

Epoch 1/60: 100%|██████████| 9/9 [00:08<00:00,  1.12it/s]


Epoch 1:
  Train Loss: 1.8517, Accuracy: 32.22%
  Val   Loss: 2.0015, Accuracy: 26.63%


Epoch 2/60: 100%|██████████| 9/9 [00:07<00:00,  1.22it/s]


Epoch 2:
  Train Loss: 1.5193, Accuracy: 47.04%
  Val   Loss: 1.5360, Accuracy: 45.56%


Epoch 3/60: 100%|██████████| 9/9 [00:07<00:00,  1.14it/s]


...

Epoch 19:
  Train Loss: 0.7868, Accuracy: 69.81%
  Val   Loss: 1.1872, Accuracy: 51.48%


Epoch 20/60: 100%|██████████| 9/9 [00:05<00:00,  1.52it/s]


Epoch 20:
  Train Loss: 0.7653, Accuracy: 70.93%
  Val   Loss: 1.1744, Accuracy: 54.44%


Epoch 21/60: 100%|██████████| 9/9 [00:06<00:00,  1.46it/s]


Epoch 21:
  Train Loss: 0.7318, Accuracy: 72.78%
  Val   Loss: 1.1611, Accuracy: 56.21%
Early stopping en época 21
Modelo guardado como 'mlp_model.pth'


In [ ]:
# %load_ext tensorboard
# !tensorboard --logdir=runs/mlp_experimento_1